In [1]:
#"Up and out option.  (Chesney at al. 1997, P.167)
#consecutive (uninterrupted) (Chesney et al. 1997, P.167)

In [7]:
# down and out Parisian Down Barrier Hitting Time.(Chesney et al. 1997, P.169)

display(Math(r'''
\tilde{H}^{-}_{L,D}(S) = \inf \left\{ t \geq 0 \,\middle|\, 
\mathbf{1}_{S_t < L} \cdot (t - g^{S}_{L,t}) \geq D \right\}
'''))

<IPython.core.display.Math object>

In [9]:
#price of the discretely monitored parisian option (Bernard and Boyle 2011, P.179)
from IPython.display import display, Math

# theoretical equation
eq1 = Math(r''' 
\text{PUIC}^{\text{d}}(T) = e^{-rT} \, \mathbb{E}_\mathbb{Q} \left[ (S_n - K)^+ \, \mathbf{1}_{\tau^d \leq T} \right]
''')

# Monte-Carlo-Approximation
eq2 = Math(r'''
\text{PUIC}^{\text{d}}(T) = e^{-rT} \cdot \frac{1}{N} \sum_{k=1}^{N} \left[ (S_n^k - K)^+ \, \mathbf{1}_{\tau_k^d \leq T} \right]
''')

display(eq1)
display(eq2)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [11]:
#Parisian Knock-out function

#Determines if a path gets knocked out under a Parisian up-and-out condition.
#asKnock-out occurs only if the underlying remains above the barrier for 'threshold_days' consecutive days.

threshold_days = 10 #Parisian Window D

def parisian_knockout(S, B, threshold_days):
  
    n_sim, n_steps = S.shape
    knocked_out = np.zeros(n_sim, dtype=bool)

    for i in range(n_sim):
        consecutive = 0   #counts the length of the current "Excursion" above B.
        for t in range(n_steps):   #The current price S_{i,t} is above the barrier --> Excursion continues → count days.
            if S[i, t] >= B:
                consecutive += 1
                if consecutive >= threshold_days:
                    knocked_out[i] = True
                    break
            else:
                consecutive = 0  # Reset if price falls below barrier

    return knocked_out


In [13]:
#Full Code discretly monitored up and out call parisian option with antithetical variance reduction 
import time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Math 
import seaborn as sns
import scipy.stats as stats

np.random.seed(42)

#discretly monitored up and out call barrier option
# Model Parameters 
S0 = 100        # Start Price
K = 100         # Strike
L = 120         # Up-and-Out Barrier
T = 1.0         # Duration (In Years)
r = 0.05        # risk-free interst rate
sigma = 0.2     # Volatility
dividend  = 0.03   # Dividend yield (continuous)

# Monte Carlo Parameter
n_sim = 100000       # Number of simulations
n_steps = 252       # Number of time discretisations (1 day)
dt = T / n_steps    # Timestep
D = 10 #Parisian time-window D


#GBM in python code with antithetic variates
Z = np.random.normal(size=(n_sim, n_steps)) #matrix with standard-normally distributed random numbers for use in wiener-process
Z_anti = -Z  # antithetic counterpart

S = np.zeros((n_sim, n_steps)) #structure for all the simulated price history
S_anti = np.zeros((n_sim, n_steps))

S[:, 0] = S0 #every first column of each simulation starts with value S_0
S_anti[:, 0] = S0


for t in range(1, n_steps):
    S[:, t] = S[:, t - 1] * np.exp((r - dividend - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z[:, t]) #GBM equation with dividend 
    S_anti[:, t] = S_anti[:, t - 1] * np.exp((r - dividend - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z_anti[:, t]) #GBM equation with dividend                                                                                                  #Dividdens from Glassman, 2003, P.97

#Parisian Knock-out function

#Determines if a path gets knocked out under a Parisian up-and-out condition.
#asKnock-out occurs only if the underlying remains above the barrier for 'threshold_days' consecutive days.


def parisian_knock(S, L, D):
  
    n_sim, n_steps = S.shape
    knock = np.zeros(n_sim, dtype=bool)

    for i in range(n_sim):
        consecutive = 0   #counts the length of the current "Excursion" above L
        for t in range(n_steps):   #The current price S_{i,t} is above the barrier --> Excursion continues → count days.
            if S[i, t] >= L:
                consecutive += 1
                if consecutive >= D:
                    knock[i] = True
                    break
            else:
                consecutive = 0  # Reset if price falls below barrier

    return knocked


knock = parisian_knock(S, L, D)

knockout_anti = parisian_knockout(S_anti, B, threshold_days)


#python code for payoff and discounted price
payoff = np.where(knock, 0.0, np.maximum(S[:, -1] - K, 0.0))
payoff_anti = np.where(knockout_anti, 0.0, np.maximum(S_anti[:, -1] - K, 0.0))

# Average both payoffs for variance reduction
payoff_avg = 0.5 * (payoff + payoff_anti)

# Discounted Price (Antithetic Estimator)
discounted_price = np.exp(-r * T) * np.mean(payoff_avg)

#stats
all_discounted = np.exp(-r * T) * payoff_avg  #Calculate all discounted individual payoffs
std_dev = np.std(all_discounted, ddof=1)   # standard deviation
standard_error = std_dev / np.sqrt(len(all_discounted))  # Standard error

z = stats.norm.ppf(0.95) #95% confidence interval
ci_low = discounted_price - z * standard_error
ci_high = discounted_price + z * standard_error

#print
# Output
print (f"Discountet Price: {discounted_price:.4f}")
print(f"Standard Deviation of Payoffs: {std_dev:.4f}")
print(f"Standard Error of the Estimator: {standard_error:.6f}")
print(f"95% Confidence Interval: [{ci_low:.4f}, {ci_high:.4f}]")


Discountet Price: 2.0905
Standard Deviation of Payoffs: 2.9105
Standard Error of the Estimator: 0.029105
95% Confidence Interval: [2.0427, 2.1384]


In [14]:
# count Knockouts
num_knockout_parisian = np.sum(knockout)
num_knockout_parisian_anti = np.sum(knockout_anti)

total_knockout_parisian = num_knockout_parisian + num_knockout_parisian_anti
total_paths = len(knockout) + len(knockout_anti)


print(f"Knock-outs        {total_knockout_parisian} from {total_paths} Paths")
print(f"Knock-out-Rate    {100 * total_knockout_parisian / total_paths:.2f}%")

Knock-outs        5106 from 20000 Paths
Knock-out-Rate    25.53%


In [2]:
#up
if S[i, t] >= B:

#down
if S[i, t] <= B:

IndentationError: expected an indented block after 'if' statement on line 2 (1915086471.py, line 5)

In [4]:
#out call
payoff = np.where(knockout, 0.0, np.maximum(S[:, -1] - K, 0.0))
payoff_anti = np.where(knockout_anti, 0.0, np.maximum(S_anti[:, -1] - K, 0.0))

#in call
payoff = np.where(knockout, np.maximum(S[:, -1] - K, 0.0), 0.0)
payoff_anti = np.where(knockout_anti, np.maximum(S_anti[:, -1] - K, 0.0), 0.0)

NameError: name 'np' is not defined

In [ ]:
import numpy as np

# Parameters
S0 = 100           # Initial stock price
K = 100            # Strike price
T = 1.0            # Time to maturity
r = 0.05           # Risk-free rate
sigma = 0.2        # Volatility
L = 90             # Barrier level
D = 0.1            # Minimum time below barrier to knock out
n_paths = 5000     # Half the number of paths (antithetic doubles it)
n_steps = 252      # Daily steps
dt = T / n_steps

# Drift under Girsanov's theorem
m = (r - 0.5 * sigma**2) / sigma
t_grid = np.linspace(dt, T, n_steps)
discount_factor = np.exp((r + 0.5 * m**2) * T)

# Generate standard normal random numbers
Z_random = np.random.normal(0, np.sqrt(dt), size=(n_paths, n_steps))
Z_antithetic = -Z_random  # Antithetic counterpart

# Combine both sets
Z_all = np.concatenate([Z_random, Z_antithetic], axis=0)

# Create Brownian motion with drift: Z_t = W_t + m * t
Z_paths = np.cumsum(Z_all, axis=1)
Z_paths += m * t_grid  # Apply drift

# Convert to stock prices: S_t = S0 * exp(Z_t)
S_paths = S0 * np.exp(Z_paths)

# Determine knock-out paths
below_barrier = S_paths < L
time_below = np.zeros_like(below_barrier, dtype=float)
knocked_out = np.zeros(2 * n_paths, dtype=bool)

for i in range(1, n_steps):
    time_below[:, i] = np.where(below_barrier[:, i], time_below[:, i-1] + dt, 0)
    knocked_out |= time_below[:, i] >= D

# Compute payoff φ(ST) = max(ST - K, 0)
ST = S_paths[:, -1]
payoffs = np.maximum(ST - K, 0)

# Apply knock-out condition: 1_{H_{b,D} > T}
payoffs[knocked_out] = 0

# Final Z_T for correction term: exp(-m * Z_T)
Z_T = Z_paths[:, -1]
adjusted_payoffs = payoffs * np.exp(-m * Z_T)

# Monte Carlo price using antithetic variates
price = discount_factor * np.mean(adjusted_payoffs)

print(f"Monte Carlo price (with antithetic variates): {price:.4f}")